In [ ]:
from google.colab import drive
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from tqdm import tqdm

import pandas as pd
import re
import string

file_path = '/content/drive/MyDrive/DataSet/final_merge_dataset.csv'
df = pd.read_csv(file_path)
df.head(10)

,Judul,Waktu,Link,Content,tag1,tag2,tag3,tag4,tag5,source
0,"Viral Isu PHK Buruh Gudang Garam, Said Iqbal: ...",6 September 2025,https://nasional.kompas.com/read/2025/09/06/14...,"JAKARTA, KOMPAS.com – Presiden Konfederasi Se...",Said Iqbal,industri rokok,PT Gudang Garam,PHK massal,phk massal 2025 terbaru,kompas
1,"Gempa M 5,3 Guncang Pulau Doi Maluku Utara","Senin, 12 Agu 2024 21:58 WIB",https://news.detik.com/berita/d-7486691/gempa-...,"Gempa bumi berkekuatan magnitudo (M) 5,3 mengg...",pulau doi,gempa,NaN,NaN,NaN,detik
2,"Toko Emas Palsu di Riau Dibongkar Polisi, Perh...","Rabu, 30 Jul 2025 22:22 WIB",https://news.detik.com/melindungi-tuah-marwah/...,Satreskrim Polres Bengkalis membongkar praktik...,pemalsuan emas,emas palsu,polres bengkalis,polda riau,melindungi tuah marwah,detik
3,Minyakita Tak Sesuai Ukuran juga Ditemukan di ...,"Senin, 10 Mar 2025 23:15 WIB",https://news.detik.com/berita/d-7816829/minyak...,Polisi mendatangi salah satu gudang Minyakita ...,minyakita,kudus,NaN,NaN,NaN,detik
4,"Pimpin LDP, Sanae Takaichi Calon Kuat PM Perem...",4 Oktober 2025 | 14.00 WIB,https://www.tempo.co/internasional/pimpin-ldp-...,"Baca berita dengan sedikit iklan, klik di sin...",jepang,perdana-menteri,sanae-takaichi,perempuan,ldp,tempo
5,Gubernur Banten Terpilih Andra Soni Temui Pres...,"Jumat, 13 Des 2024 22:12 WIB",https://news.detik.com/berita/d-7685816/gubern...,Presiden Prabowo Subianto bertemu dengan Guber...,prabowo subianto,andra soni,istana negara,NaN,NaN,detik
6,Yusril: Pedemo Ditangkap Polisi karena Lakukan...,27 September 2025 | 08.29 WIB,https://www.tempo.co/hukum/yusril-pedemo-ditan...,"Baca berita dengan sedikit iklan, klik di sin...",yusril,demonstrasi,demonstran,penangkapan-aktivis,yusril-ihza-mahendra,tempo
7,"Banjir Besar Landa Jabodetabek, Pimpinan MPR I...",4 Maret 2025,https://nasional.kompas.com/read/2025/03/04/22...,"JAKARTA, KOMPAS.com - Wakil Ketua MPR RI Eddy...",eddy soeparno,mitigasi bencana,banjir jabodetabek,krisis iklim,Banjir Jabodetabek hari ini,kompas
8,"Lagu ""Indonesia Raya"" Kena Royalti? Istana Men...",15 Agustus 2025,https://nasional.kompas.com/read/2025/08/15/18...,"JAKARTA, KOMPAS.com - Menteri Sekretaris Nega...",royalti,istana,Indonesia Raya,PSSI,Prasetyo Hadi,kompas
9,Remaja Ditindak Polisi di Serpong karena Bawa ...,29 April 2025,https://megapolitan.kompas.com/read/2025/04/29...,"TANGERANG SELATAN, KOMPAS.com - Seorang remaj...",polisi,remaja,Mobil berpelat asing,Remaja ditindak polisi,Gunakan mobil pelat asing,kompas


In [ ]:
JUMLAH_DATA = 1000

print(f"Sedang mengambil {JUMLAH_DATA} data teratas...")
df_proses = df.head(JUMLAH_DATA).copy()

# Gabungkan Judul dan Content
df_proses['Dokumen_Asli'] = df_proses['Judul'] + " " + df_proses['Content']

# Buang data kosong
df_proses = df_proses.dropna(subset=['Dokumen_Asli'])

print(f"Siap memproses {len(df_proses)} dokumen!")

Sedang mengambil 1000 data teratas...
Siap memproses 1000 dokumen!


In [ ]:
# Siapkan tools Sastrawi
factory_sw = StopWordRemoverFactory()
stopword_remover = factory_sw.create_stop_word_remover()

factory_stem = StemmerFactory()
stemmer = factory_stem.create_stemmer()

def proses_teks(text):
    # 1. Case Folding (Huruf kecil semua)
    text = str(text).lower()

    # 2. Cleaning (Hapus angka dan tanda baca)
    text = re.sub(r"\d+", "", text) # Hapus angka
    text = text.translate(str.maketrans("", "", string.punctuation)) # Hapus tanda baca
    text = re.sub(r'\s+', ' ', text).strip() # Hapus spasi berlebih

    # 3. Stopword Removal (Hapus kata hubung: dan, yang, di)
    text = stopword_remover.remove(text)

    # 4. Stemming (Ubah ke kata dasar) - Bagian paling lama
    text = stemmer.stem(text)

    return text

print("Fungsi preprocessing siap digunakan!")

Fungsi preprocessing siap digunakan!


In [ ]:
tqdm.pandas()

print("Sedang memproses teks (Stemming)... Mohon tunggu.")
# Kita terapkan fungsi tadi ke kolom 'Dokumen_Asli'
df_proses['Cleaned_Content'] = df_proses['Dokumen_Asli'].progress_apply(proses_teks)

print("Selesai!")

Sedang memproses teks (Stemming)... Mohon tunggu.


100%|██████████| 1000/1000 [27:31<00:00,  1.65s/it]

Selesai!


In [ ]:
# Cell 5: Cek & Simpan
# Tampilkan perbandingan sebelum dan sesudah
print(df_proses[['Dokumen_Asli', 'Cleaned_Content']].head(3))

# Simpan ke Google Drive
output_path = '/content/drive/MyDrive/DataSet/indonesia_news_1000.csv'
df_proses.to_csv(output_path, index=False)

print(f"File berhasil disimpan di: {output_path}")

                                        Dokumen_Asli  \
0  Viral Isu PHK Buruh Gudang Garam, Said Iqbal: ...   
1  Gempa M 5,3 Guncang Pulau Doi Maluku Utara Gem...   
2  Toko Emas Palsu di Riau Dibongkar Polisi, Perh...   

                                     Cleaned_Content  
0  viral isu phk buruh gudang garam said iqbal su...  
1  gempa m guncang pulau doi malu utara gempa bum...  
2  toko emas palsu riau bongkar polisi hias kg si...  
File berhasil disimpan di: /content/drive/MyDrive/DataSet/indonesia_news_1000.csv
